# Video Pipeline Details

This notebook goes into detail about the stages of the video pipeline in the base overlay and is written for people who want to create and integrate their own video IP. For most regular input and output use cases the high level wrappers of `HDMIIn` and `HDMIOut` should be used.

Both the input and output pipelines in the base overlay consist of four stages, an HDMI frontend, a colorspace converter, a pixel format converter, and the video DMA. For the input the stages are arranged Frontend -> Colorspace Converter -> Pixel Format -> VDMA with the order reversed for the output side. The aim of this notebook is to give you enough information to use each stage separately and be able to modify the pipeline for your own ends.

Before exploring the pipeline we'll import the entire pynq.lib.video module where all classes relating to the pipelines live. We'll also load the base overlay to serve as an example.

The following table shows the IP responsible for each stage in the base overlay which will be referenced throughout the rest of the notebook

|Stage             | Input IP                               | Output IP                          |
|------------------|:---------------------------------------|:-----------------------------------|
|Frontend (Timing) |`video/hdmi_in/frontend/vtc_in`         |`video/hdmi_out/frontend/vtc_out`   |
|Frontend (Other)  |`video/hdmi_in/frontend/axi_gpio_hdmiin`|`video/hdmi_out/frontend/axi_dynclk`|
|Colour Space      |`video/hdmi_in/color_convert`           |`video/hdmi_out/color_convert`      |
|Pixel Format      |`video/hdmi_in/pixel_pack`              |`video/hdmi_outpixel_unpack`        |
|VDMA              |`video/axi_vdma`                        |`video/axi_vdam`                    |

## HDMI Frontend

The HDMI frontend modules wrap all of the clock and timing logic. The HDMI input frontend can be used independently from the rest of the pipeline by accessing its driver from the base overlay.

Creating the device will signal to the computer that a monitor is connected. Starting the frontend will wait attempt to detect the video mode, blocking until a lock can be achieved. Once the frontend is started the video mode will be available.

The HDMI output frontend can be accessed in a similar way.

and the mode must be set prior to starting the output. In this case we are just going to use the same mode as the input.

Note that nothing will be displayed on the screen as no video data is currently being send.

## Colorspace conversion

The colorspace converter operates on each pixel independently using a 3x4 matrix to transform the pixels. The converter is programmed with a list of twelve coefficients in the folling order:

|     |in1 |in2 |in3 | 1  |
|-----|----|----|----|----|
|out1 |c1  |c2  |c3  |c10 |
|out2 |c4  |c5  |c6  |c11 |
|out3 |c7  |c8  |c9  |c12 |

Each coefficient should be a floating point number between -2 and +2.

The pixels to and from the HDMI frontends are in BGR order so a list of coefficients to convert from the input format to RGB would be:

    [0, 0, 1,
     0, 1, 0,
     1, 0, 0,
     0, 0, 0]

reversing the order of the pixels and not adding any bias.

The driver for the colorspace converters has a single property that contains the list of coefficients.

## Pixel format conversion

The pixel format converters convert between the 24-bit signal used by the HDMI frontends and the colorspace converters to either an 8, 24, or 32 bit signal. 24-bit mode passes the input straight through, 32-bit pads the additional pixel with 0 and 8-bit mode selects the first channel in the pixel. This is exposed by a single property to set or get the number of bits.

## Video DMA

The final element in the pipeline is the video DMA which transfers video frames to and from memory. The VDMA consists of two channels, one for each direction which operate completely independently. To use a channel its mode must be set prior to start being called. After the DMA is started `readframe` and `writeframe` transfer frames. Frames are only transferred once with the call blocking if necessary. `asyncio` coroutines are available as `readframe_async` and `writeframe_async` which yield instead of blocking. A frame of the size of the output can be retrieved from the VDMA by calling `writechannel.newframe()`. This frame is not guaranteed to be initialised to blank so should be completely written before being handed back.

In this case, because we are only using 8 bits per pixel, only the red channel is read and displayed.

The two channels can be tied together which will ensure that the input is always mirrored to the output

In [1]:
# Full restart
from pynq.overlays.base import BaseOverlay
from pynq import MMIO
import os, datetime
from pynq.lib.video import *
from IPython.display import clear_output
import time

base = BaseOverlay("base.bit")

# HDMI frontend
hdmiin_frontend = base.video.hdmi_in.frontend
hdmiin_frontend.start()

hdmiout_frontend = base.video.hdmi_out.frontend
hdmiout_frontend.mode = VideoMode(width=1920, height=1080, bits_per_pixel=24, fps=240)
hdmiout_frontend.start()

# Colorspace -- pass through (no BGR->RGB swap needed for 24-bit)
colorspace_in  = base.video.hdmi_in.color_convert
colorspace_out = base.video.hdmi_out.color_convert
passthrough = [1, 0, 0,
               0, 1, 0,
               0, 0, 1,
               0, 0, 0]
colorspace_in.colorspace  = passthrough
colorspace_out.colorspace = passthrough

# 24-bit pixel format
pixel_in  = base.video.hdmi_in.pixel_pack
pixel_out = base.video.hdmi_out.pixel_unpack
pixel_in.bits_per_pixel  = 24
pixel_out.bits_per_pixel = 24

vdma = base.video.axi_vdma
in_width        = hdmiin_frontend.mode.width
in_height       = hdmiin_frontend.mode.height
out_width       = 1920
out_height      = 1080
bytes_per_pixel = 3       # 24-bit RGB
stride          = in_width * bytes_per_pixel  # full 4K row in bytes

print(f"in_width: {in_width}, in_height: {in_height}")
print(f"stride: {stride}")

vdma.writechannel.mode = VideoMode(in_width,  in_height, 24)
vdma.readchannel.mode  = VideoMode(out_width, out_height, 24)

vdma.writechannel.start()
frame     = vdma.writechannel._cache.getframe()
frame.flush()
base_addr = frame.physical_address
print(f"fresh base_addr: {hex(base_addr)}")

vdma.readchannel.start()

vdma.writechannel._mmio.write(0xAC, base_addr)
vdma.writechannel._mmio.write(0xA8, in_width  * bytes_per_pixel)
vdma.writechannel._mmio.write(0xA4, in_width  * bytes_per_pixel)
vdma.writechannel._mmio.write(0xA0, in_height)

ctrl  = vdma.readchannel._mmio.read(0x30)
ctrl &= ~0xFF00
ctrl |= (0x01 << 8) | (1 << 12)
vdma.readchannel._mmio.write(0x30, ctrl)
time.sleep(1.0)

CTRL_REG       = 0x00
BASE_ADDR_REG  = 0x04
OUT_WIDTH_REG  = 0x08
OUT_HEIGHT_REG = 0x0C
BPP_REG        = 0x10
STRIDE_REG     = 0x14
VDMA_ADDR_REG  = 0x18
STATUS_REG     = 0x1C
TRIG_REG       = 0x20

qs_addr        = base.ip_dict['quadrant_switcher_1']['phys_addr']
vdma_phys_addr = base.ip_dict['video/axi_vdma']['phys_addr']
qs = MMIO(qs_addr, 0x1000)

qs.write(CTRL_REG,       0)
qs.write(BASE_ADDR_REG,  base_addr)
qs.write(OUT_WIDTH_REG,  out_width)
qs.write(OUT_HEIGHT_REG, out_height)
qs.write(BPP_REG,        bytes_per_pixel)
qs.write(STRIDE_REG,     stride)
qs.write(VDMA_ADDR_REG,  vdma_phys_addr)
qs.write(CTRL_REG, 1)
print("\nQS enabled")

# qs.write(CTRL_REG, 0)  # disable
# time.sleep(0.5)
# # Now only one LED should be on (whichever quadrant it stopped at)
# raw = qs.read(STATUS_REG)
# q = raw & 0x3
# print(f"Stopped at Q={q}, only LED{q} should be on")

print(f"bytes_per_pixel: {bytes_per_pixel}")
print(f"stride:          {stride}")
print(f"stride reg:      {qs.read(STRIDE_REG)}")
print(f"Q0: {hex(base_addr)}")
print(f"Q1: {hex(base_addr + out_width * bytes_per_pixel)}")
print(f"Q2: {hex(base_addr + out_height * stride)}")
print(f"Q3: {hex(base_addr + out_height * stride + out_width * bytes_per_pixel)}")

in_width: 1920, in_height: 1080
stride: 5760
fresh base_addr: 0x60900000

QS enabled
bytes_per_pixel: 3
stride:          5760
stride reg:      5760
Q0: 0x60900000
Q1: 0x60901680
Q2: 0x60eeec00
Q3: 0x60ef0280


In [10]:
print(f"MM2S ctrl:   {hex(vdma.readchannel._mmio.read(0x30))}")
print(f"MM2S status: {hex(vdma.readchannel._mmio.read(0x34))}")
print(f"MM2S addr:   {hex(vdma.readchannel._mmio.read(0x5C))}")
print(f"MM2S hsize:  {hex(vdma.readchannel._mmio.read(0x54))}")
print(f"MM2S vsize:  {hex(vdma.readchannel._mmio.read(0x50))}")
print(f"MM2S stride: {hex(vdma.readchannel._mmio.read(0x58))}")

MM2S ctrl:   0x10082
MM2S status: 0x10001
MM2S addr:   0x0
MM2S hsize:  0x0
MM2S vsize:  0x0
MM2S stride: 0x0


In [6]:
gpio_leds_addr = base.ip_dict['gpio_leds']['phys_addr']
gl = MMIO(gpio_leds_addr, 0x1000)
gl.write(0x00, 0xF)

print(base.ip_dict.keys())

dict_keys(['video/hdmi_in/frontend', 'video/hdmi_out/color_convert', 'video/hdmi_out/frontend', 'gpio_sws', 'fmch_axi_iic', 'video/axi_vdma', 'axi_intc_0', 'reset_control', 'gpio_btns', 'gpio_leds', 'quadrant_switcher_1', 'video/hdmi_in/color_convert', 'shutdown_HP0_FPD', 'video/hdmi_in/pixel_pack', 'video/hdmi_out/pixel_unpack', 'shutdown_HP2_FPD', 'shutdown_LPD', 'video/phy/vid_phy_controller', 'ps_e_0'])


In [4]:
import os, datetime
bit_path = "/usr/local/share/pynq-venv/lib/python3.10/site-packages/pynq/overlays/base/base.bit"
print(datetime.datetime.fromtimestamp(os.path.getmtime(bit_path)))

2026-04-12 20:39:44.301033


In [2]:
# Frame Assembler - 4x 1080p input → 1x 4K output
from pynq.overlays.base import BaseOverlay
from pynq import MMIO, allocate
from pynq.lib.video import *
from IPython.display import clear_output
import numpy as np
import time

base = BaseOverlay("base.bit")

# HDMI frontend
hdmiin_frontend = base.video.hdmi_in.frontend
hdmiin_frontend.start()

hdmiout_frontend = base.video.hdmi_out.frontend
hdmiout_frontend.mode = VideoMode(width=3840, height=2160, bits_per_pixel=24, fps=60)
hdmiout_frontend.start()

# Colorspace passthrough
passthrough = [0,0,1, 
               0,1,0, 
               1,0,0, 
               0,0,0]
base.video.hdmi_in.color_convert.colorspace  = passthrough
base.video.hdmi_out.color_convert.colorspace = passthrough

# Pixel format
base.video.hdmi_in.pixel_pack.bits_per_pixel  = 24
base.video.hdmi_out.pixel_unpack.bits_per_pixel = 24

# Dimensions
in_width        = 1920
in_height       = 1080
out_width       = 3840
out_height      = 2160
bytes_per_pixel = 3
stride          = out_width * bytes_per_pixel   # 11520 bytes per 4K row

vdma = base.video.axi_vdma
vdma.writechannel.mode = VideoMode(in_width,  in_height, 24)  # S2MM: 1080p capture
vdma.readchannel.mode  = VideoMode(out_width, out_height, 24) # MM2S: 4K readout

# Allocate 4K framebuffer (assembler writes here, MM2S reads from here)
frame_4k      = allocate(shape=(out_height, out_width, 3), dtype=np.uint8)
frame_4k[:]   = 0
frame_4k.flush()
base_addr     = frame_4k.physical_address
print(f"base_addr: {hex(base_addr)}")
print(f"stride:    {stride}")



# Start S2MM first, then MM2S after buffer is partially filled
vdma.writechannel.start()
time.sleep(1/60 * 3)
vdma.readchannel.start()

# Point MM2S at the 4K buffer and start
vdma.readchannel._mmio.write(0x5C, base_addr)
vdma.readchannel._mmio.write(0x58, stride)
vdma.readchannel._mmio.write(0x54, out_width * bytes_per_pixel)
vdma.readchannel._mmio.write(0x50, out_height)


# Enable S2MM IOC interrupt (drives s2mm_introut → frame_assembler frame_done)
s2mm_ctrl  = vdma.writechannel._mmio.read(0x30)
s2mm_ctrl &= ~(1 << 1)   # clear Circular_Park bit → park mode
s2mm_ctrl &= ~0xFF00
s2mm_ctrl |= (0x01 << 8) | (1 << 12)
vdma.writechannel._mmio.write(0x30, s2mm_ctrl)

# Set park frame to 0 so it uses Start_Address_1 (0xAC)
vdma.writechannel._mmio.write(0x28, 0x00000000)  # S2MM_PARK_PTR_REG frame=0
time.sleep(0.1)

# Frame assembler registers
CTRL_REG      = 0x00
BASE_ADDR_REG = 0x04
IN_WIDTH_REG  = 0x08
IN_HEIGHT_REG = 0x0C
BPP_REG       = 0x10
STRIDE_REG    = 0x14
VDMA_ADDR_REG = 0x18
STATUS_REG    = 0x1C
TRIG_REG      = 0x20

fa_addr        = base.ip_dict['frame_assembler_0']['phys_addr']
vdma_phys_addr = base.ip_dict['video/axi_vdma']['phys_addr']
fa = MMIO(fa_addr, 0x1000)

fa.write(CTRL_REG,      0)
fa.write(BASE_ADDR_REG, base_addr)
fa.write(IN_WIDTH_REG,  in_width)
fa.write(IN_HEIGHT_REG, in_height)
fa.write(BPP_REG,       bytes_per_pixel)
fa.write(STRIDE_REG,    stride)
fa.write(VDMA_ADDR_REG, vdma_phys_addr)
fa.write(CTRL_REG,      1)
print("FA enabled")

print(f"\nQuadrant addresses:")
print(f"  Q0: {hex(base_addr)}")
print(f"  Q1: {hex(base_addr + in_width * bytes_per_pixel)}")
print(f"  Q2: {hex(base_addr + in_height * stride)}")
print(f"  Q3: {hex(base_addr + in_height * stride + in_width * bytes_per_pixel)}")

# Monitor
state_names = {0:"IDLE",1:"CALC",2:"WR_BOTH",3:"WR_ADDR",
               4:"WR_DATA",5:"WR_RESP",6:"NEXT",7:"DONE"}

base_addr: 0x68100000
stride:    11520
FA enabled

Quadrant addresses:
  Q0: 0x68100000
  Q1: 0x68101680
  Q2: 0x68cdd800
  Q3: 0x68cdee80


### Frame Ownership

The VDMA driver has a strict method of frame ownership. Any frames returned by `readframe` or `newframe` are owned by the user and should be destroyed by the user when no longer needed by calling `frame.freebuffer()`. Frames handed back to the VDMA with `writeframe` are no longer owned by the user and should not be touched - the data may disappear at any time.

In [6]:
print(f"base_addr:   {hex(base_addr)}")
print(f"S2MM addr:   {hex(vdma.writechannel._mmio.read(0xAC))}")
print(f"buffer end:  {hex(base_addr + out_height * stride)}")
print()
print(f"Q0 expected: {hex(base_addr)}")
print(f"Q1 expected: {hex(base_addr + in_width * bytes_per_pixel)}")
print(f"Q2 expected: {hex(base_addr + in_height * stride)}")
print(f"Q3 expected: {hex(base_addr + in_height * stride + in_width * bytes_per_pixel)}")
print()
# Check if writechannel's own buffer is interfering
print(f"writechannel buffer: {hex(vdma.writechannel._cache.getframe().physical_address)}")


base_addr:   0x72900000
S2MM addr:   0x734dd800
buffer end:  0x740bb000

Q0 expected: 0x72900000
Q1 expected: 0x72901680
Q2 expected: 0x734dd800
Q3 expected: 0x734dee80

writechannel buffer: 0x68d00000


## Cleaning up

It is vital to stop the VDMA before reprogramming the bitstream otherwise the memory system of the chip can be placed into an undefined state. If the monitor does not power on when starting the VDMA this is the likely cause.

In [2]:
vdma.readchannel.stop()
vdma.writechannel.stop()

In [12]:
frame_4k.flush()
frame_4k.freebuffer()
del frame_4k


NameError: name 'frame_4k' is not defined